# V2 · ViT Figure-Type Classification (SPIQA)
Vision track, model 2. Fine-tunes `google/vit-base-patch16-224-in21k` to classify scientific figures into
**plot, schematic, photo, table, other**. Baselines: ViT linear probe (frozen backbone) and ResNet-50 (CNN).

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Add `HF_TOKEN` in the Secrets panel (key icon) with notebook access on.
Edit the two settings below once.

In [ ]:
REPO_URL = "https://github.com/mkavinvelavan/researchlens.git"   # <- change
HF_USERNAME = "YOUR_HF_USERNAME"                                          # <- change
DRIVE_ROOT = "/content/drive/MyDrive/ResearchLens"                        # checkpoints + results live here

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.exists("/content/researchlens"):
    !git clone -q {REPO_URL} /content/researchlens
else:
    !git -C /content/researchlens pull -q
%cd /content/researchlens
!pip install -q -r requirements-colab.txt

os.environ["HF_USERNAME"] = HF_USERNAME
os.environ["RL_DRIVE_ROOT"] = DRIVE_ROOT
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build the dataset from SPIQA
Downloads the SPIQA JSON + image zips from the Hub to the local Colab disk (not Drive — much faster),
then samples up to `max_per_class` training figures per class. The first run takes a while (large zip).

In [ ]:
!python -m researchlens.vision.prepare_spiqa_figures --out_dir /content/data/spiqa_figs --max_per_class 3000

## 2. Train ViT (3 seeds; seed 42 pushed to the Hub)

In [ ]:
SEEDS = [42, 43, 44]
for s in SEEDS:
    push = "--push_to_hub" if s == 42 else ""
    !python -m researchlens.vision.train_figcls --config configs/v2_figcls.yaml --seed {s} {push}

## 3. Baselines: linear probe + ResNet-50

In [ ]:
for s in SEEDS:
    !python -m researchlens.vision.train_figcls --config configs/v2_figcls.yaml --freeze_backbone --run_name vit_linear_probe --seed {s}
for s in SEEDS:
    !python -m researchlens.vision.train_figcls --config configs/v2_figcls.yaml --model_name microsoft/resnet-50 --run_name resnet50 --seed {s}

## 4. Attention maps + misclassified examples (needs the images, so run in this session)

In [ ]:
R = f"{DRIVE_ROOT}/results/v2_figcls/vit_base/seed_42"
M = f"{DRIVE_ROOT}/models/v2_figcls/vit_base/seed_42"
!python -m researchlens.vision.explain --results_dir {R} --model_dir {M} --data_dir /content/data/spiqa_figs

## 5. Report: all graphs + tables

In [ ]:
from researchlens.evaluation.reports import figcls_report
REPORT_DIR = f"{DRIVE_ROOT}/results/v2_figcls/report"
display(figcls_report(f"{DRIVE_ROOT}/results"))

from pathlib import Path
from IPython.display import Image, display
for f in sorted(Path(REPORT_DIR).glob("*.png")):
    print(f.name); display(Image(str(f), width=650))

for f in sorted(Path(R, "figures").glob("*.png")):
    print(f.name); display(Image(str(f), width=800))